### Binary Constants

Following Python function is from the course notes:

In [ ]:
def runpywasm(wasmfile):
    from pywasm.core import Machine, Runtime, FuncType, ValType
    # P0lib implementation in Python
    def write(_: Machine, args: list[int]) -> list[int]:
        print(args[0]); return []
    def writeln(_: Machine, args: list[int]) -> list[int]:
        print(); return []
    def read(_: Machine, args: list[int]) -> list[int]:
        return [int(input())]
    # Create runtime
    runtime = Runtime()
    runtime.imports = {'P0lib':
        {'write': runtime.allocate_func_host(FuncType([ValType.i32()], []), write),
         'writeln': runtime.allocate_func_host(FuncType([], []), writeln),
         'read': runtime.allocate_func_host(FuncType([], [ValType.i32()]), read)}}
    # Create and run instance
    instance = runtime.instance_from_file(wasmfile)

In [ ]:
import import_ipynb
from P0 import compileString

Modify the attached P0 compiler to allow for binary constants. For this, the grammar of `number` should be:

    number ::= digit {digit} | '0' 'b' binarydigit {binarydigit}
    digit ::= '0' | ... | '9'
    binarydigit ::= '0' | '1'

State which parts of the compiler need to be modified. A test case is below.

Only the scanner needs to change — specifically the `number()` procedure in `SC.ipynb`. Everything downstream (parser `P0.ipynb`, symbol table `ST.ipynb`, code generators `CGast` / `CGwat` / `CGriscv` / `CGmips`) consumes `NUMBER` tokens as 32-bit integer values and is agnostic to the source representation, so extending the integer literal's surface syntax is a scanner-only change.

**Modified `number()` in `SC.ipynb`:**

```python
def number():
    global sym, val
    sym, val = NUMBER, 0
    if ch == '0':
        getChar()
        if ch == 'b':
            getChar()
            if ch not in ('0', '1'): mark('binary digit expected')
            while ch in ('0', '1'):
                val = 2 * val + int(ch)
                getChar()
        else:
            while '0' <= ch <= '9':
                val = 10 * val + int(ch)
                getChar()
    else:
        while '0' <= ch <= '9':
            val = 10 * val + int(ch)
            getChar()
    if val >= 2**31:
        mark('number too large'); val = 0
```

With this patch, `compileString` on the test program below emits a valid `bin.wat` / riscv sequence. For the test case `if 0b010101 = b then write(0b00111) else write(0b11100)` with `b = 0b101010 = 42`, `0b010101 = 21 ≠ 42`, so the else branch fires and writes `0b11100 = 28`.


In [ ]:
compileString("""
program hex
    const b = 0b101010
    if 0b010101 = b then write(0b00111) else write(0b11100)
""", "bin.wat")

In [ ]:
!wat2wasm bin.wat

In [ ]:
runpywasm("bin.wasm")